# Consumer Shopping Behavior Analysis

Exploratory analysis of consumer shopping survey responses using pandas, NumPy, Matplotlib and Seaborn.

**Dataset required:** `data/consumer_shopping_behavior_survey.csv`. The dataset was not included with the notebook upload; add the original CSV at that path before running the analysis.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)


In [ ]:
df = pd.read_csv('../data/consumer_shopping_behavior_survey.csv')
df.head()


In [ ]:
# Initial data-quality checks
print('Shape:', df.shape)
display(df.info())
display(df.isna().sum())
print('Duplicate rows:', df.duplicated().sum())


In [ ]:
# Convert data types
df['Age'] = pd.to_numeric(df['Age'], errors='coerce')
df['Timestamp'] = pd.to_datetime(df['Timestamp'], format='%d/%m/%Y %H:%M:%S', errors='coerce')
display(df[['Age', 'Timestamp']].dtypes)
display(df['Age'].describe())


In [ ]:
# Group respondents by age
bins = [0, 18, 25, 35, 50, 100]
labels = ['Under 18', '18-25', '26-35', '36-50', '50+']
df['Age_Group'] = pd.cut(df['Age'], bins=bins, labels=labels)
display(df['Age_Group'].value_counts())


In [ ]:
# Fill selected categorical fields
cols_to_fill = ['Gender', 'Occupation_Status', 'Country_Region', 'Shop_Most_Where', 'Preferred_Platform', 'Preferred_Payment_Method']
for col in cols_to_fill:
    df[col] = df[col].fillna('Unknown')
display(df[cols_to_fill].isna().sum())


In [ ]:
# Identify rows where all six response fields are missing
last6 = ['Online_Shopping_Satisfaction_Score', 'Trust_In_Online_Reviews_Score', 'Regret_After_Purchase_Frequency', 'Reason_Prefer_Online', 'Reason_Prefer_InStore', 'Best_Shopping_Mode']
dropoff = df[last6].isna().all(axis=1)
print('Rows with all six fields missing:', int(dropoff.sum()))
display(df.loc[dropoff, 'Timestamp'].dt.date.value_counts().sort_index())


In [ ]:
# Shopping mode and in-store preference
display(pd.crosstab(df['Shop_Most_Where'], df['Reason_Prefer_InStore'], dropna=False))


In [ ]:
# Compare satisfaction, trust and regret by age group
age_summary = df.groupby('Age_Group', observed=False).agg(
    avg_satisfaction=('Online_Shopping_Satisfaction_Score', 'mean'),
    avg_trust=('Trust_In_Online_Reviews_Score', 'mean'),
    avg_regret=('Regret_After_Purchase_Frequency', 'mean'),
    respondents=('Age_Group', 'count')
)
display(age_summary)


In [ ]:
# Country-level summary for the eight most represented countries
top_countries = df['Country_Region'].value_counts().head(8).index
country_summary = df[df['Country_Region'].isin(top_countries)].groupby('Country_Region').agg(
    avg_satisfaction=('Online_Shopping_Satisfaction_Score', 'mean'),
    bnpl_regular_rate=('Uses_BNPL_Installments', lambda x: (x == 'Yes, regularly').mean()),
    respondents=('Response_ID', 'count')
)
display(country_summary)


In [ ]:
# Compare shopping patterns by occupation
occupation_summary = df.groupby('Occupation_Status').agg(
    avg_impulse=('Impulse_Purchase_Frequency', 'mean'),
    avg_discount_importance=('Discount_Importance_Score', 'mean'),
    bnpl_regular_rate=('Uses_BNPL_Installments', lambda x: (x == 'Yes, regularly').mean()),
    respondents=('Response_ID', 'count')
).round(2).sort_values('avg_impulse', ascending=False)
display(occupation_summary)


In [ ]:
# Payment method mix by shopping mode
pivot_payment = pd.pivot_table(df, index='Shop_Most_Where', columns='Preferred_Payment_Method', values='Response_ID', aggfunc='count', fill_value=0)
pivot_payment_pct = pivot_payment.div(pivot_payment.sum(axis=1), axis=0)
display(pivot_payment_pct)


In [ ]:
# Spending category mix by gender
pivot_gender_spend = pd.pivot_table(df, index='Gender', columns='Top_Spending_Category', values='Response_ID', aggfunc='count', fill_value=0)
pivot_gender_spend_pct = pivot_gender_spend.div(pivot_gender_spend.sum(axis=1), axis=0)
display(pivot_gender_spend_pct)


In [ ]:
# Visualizations
sns.histplot(df['Age'].dropna(), bins=20, kde=True)
plt.title('Age Distribution of Survey Respondents')
plt.xlabel('Age'); plt.ylabel('Count'); plt.tight_layout(); plt.show()

sns.barplot(x=age_summary.index, y=age_summary['avg_satisfaction'], palette='viridis')
plt.title('Average Online Shopping Satisfaction by Age Group')
plt.xlabel('Age Group'); plt.ylabel('Average Satisfaction Score (1-10)'); plt.ylim(0, 10); plt.tight_layout(); plt.show()

sns.heatmap(pivot_payment_pct, annot=True, cmap='Blues', fmt='.0%', cbar_kws={'label': 'Share of Respondents'})
plt.title('Preferred Payment Method by Shopping Mode')
plt.xlabel('Preferred Payment Method'); plt.ylabel('Shopping Mode'); plt.tight_layout(); plt.show()

sns.boxplot(x='Shop_Most_Where', y='Discount_Importance_Score', data=df, palette='Set2')
plt.title('Discount Importance by Shopping Mode')
plt.xlabel('Shopping Mode'); plt.ylabel('Discount Importance Score'); plt.tight_layout(); plt.show()
